# AAPL 5-minute data cleaning
Input: `aapl_5m_raw.parquet` (or `.csv`). Output: `aapl_5m_clean.parquet`.
Run the cells in order. Nothing here uses future data.

## 1. Load

In [1]:
import numpy as np
import pandas as pd
from datetime import time as dtime

path = "aapl_5m_raw.parquet"   # or "aapl_5m_raw.csv"

if path.endswith(".csv"):
    df = pd.read_csv(path)
else:
    df = pd.read_parquet(path)

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
print(df.shape)
df.head(3)

(121412, 9)


,timestamp,open,high,low,close,volume,trade_count,vwap,ts_ny
0,2020-08-03 13:30:00+00:00,104.70,105.86,104.42,105.84,97348.0,273.0,105.059171,2020-08-03 09:30:00-04:00
1,2020-08-03 13:35:00+00:00,105.91,106.34,105.79,106.18,78580.0,225.0,106.032614,2020-08-03 09:35:00-04:00
2,2020-08-03 13:40:00+00:00,106.20,107.28,106.20,106.95,83776.0,280.0,106.813144,2020-08-03 09:40:00-04:00


## 2. Sort, remove duplicates, add New York time

In [2]:
df = df.sort_values("timestamp")
df = df.drop_duplicates("timestamp")

drop_cols = [c for c in ["symbol", "ts_ny"] if c in df.columns]
df = df.drop(columns=drop_cols)

df["ts"] = df["timestamp"].dt.tz_convert("America/New_York")
df["day"] = df["ts"].dt.date
print(df.shape)

(121412, 10)


## 3. Keep regular trading hours only

In [3]:
t = df["ts"].dt.time
in_hours = (t >= dtime(9, 30)) & (t <= dtime(15, 55))
print("rows outside 09:30-15:55:", (~in_hours).sum())
df = df[in_hours].copy()

rows outside 09:30-15:55: 0


## 4. Remove incomplete days
A full day has 78 bars. A half day (day after Thanksgiving, Christmas Eve, 3 July) has 42. Any other count means missing data, so that day is dropped.

In [4]:
df["day_bars"] = df.groupby("day")["close"].transform("size")
ok_counts = [78, 42]

bad_days = df.loc[~df["day_bars"].isin(ok_counts), "day"].unique()
print("dropping days:", list(bad_days))

df = df[df["day_bars"].isin(ok_counts)].copy()
df["is_half"] = (df["day_bars"] == 42).astype(int)
df = df.reset_index(drop=True)
print(df.shape)

dropping days: [datetime.date(2021, 4, 13), datetime.date(2024, 12, 23), datetime.date(2025, 7, 3)]
(121284, 12)


## 5. Position in the day
`bars_left` is the number of bars after this one in the same day. When you label with a 6-bar horizon, rows with `bars_left < 6` have no valid label. `bar_end` is when the bar is known (a bar stamped 09:30 is only complete at 09:35), so features must use `bar_end` as the decision time.

In [5]:
df["bar_idx"] = df.groupby("day").cumcount()
df["bars_left"] = df["day_bars"] - df["bar_idx"] - 1
df["bar_end"] = df["timestamp"] + pd.Timedelta(minutes=5)
df[["ts", "bar_idx", "bars_left", "bar_end"]].head(3)

,ts,bar_idx,bars_left,bar_end
0,2020-08-03 09:30:00-04:00,0,77,2020-08-03 13:35:00+00:00
1,2020-08-03 09:35:00-04:00,1,76,2020-08-03 13:40:00+00:00
2,2020-08-03 09:40:00-04:00,2,75,2020-08-03 13:45:00+00:00


## 6. Returns that do not mix the overnight gap with the first bar

In [6]:
first = df["bar_idx"] == 0
prev_close = df["close"].shift(1)

df["gap"] = np.where(first, np.log(df["open"] / prev_close), 0.0)
df["gap"] = df["gap"].fillna(0.0)

df["ret_1"] = np.where(
    first,
    np.log(df["close"] / df["open"]),
    np.log(df["close"] / prev_close),
)
df[["ts", "close", "gap", "ret_1"]].head(3)

,ts,close,gap,ret_1
0,2020-08-03 09:30:00-04:00,105.84,0.0,0.010829
1,2020-08-03 09:35:00-04:00,106.18,0.0,0.003207
2,2020-08-03 09:40:00-04:00,106.95,0.0,0.007226


## 7. Sanity checks (these should all pass)

In [7]:
price_cols = ["open", "high", "low", "close"]

assert df[price_cols + ["volume"]].notna().all().all()
assert (df[price_cols] > 0).all().all()

bad_high = (df["high"] < df[["open", "close"]].max(axis=1)).sum()
bad_low = (df["low"] > df[["open", "close"]].min(axis=1)).sum()
print("high/low violations:", bad_high, bad_low)
print("zero-volume bars:", (df["volume"] == 0).sum())
print("duplicate timestamps:", df["timestamp"].duplicated().sum())
print("days:", df["day"].nunique())
print("bars per day:")
print(df.groupby("day").size().value_counts())

high/low violations: 0 0
zero-volume bars: 0
duplicate timestamps: 0
days: 1560
bars per day:
78    1549
42      11
Name: count, dtype: int64


## 8. Look at the biggest moves (review only, nothing is removed)
Big gaps are usually real (earnings, tariffs). Check any you do not recognise against the news or yfinance.

In [8]:
inside = df[df["bar_idx"] > 0].copy()
inside["a"] = inside["ret_1"].abs()
print("largest moves inside a day:")
print(inside.nlargest(8, "a")[["ts", "close", "ret_1"]].to_string())

opens = df[df["bar_idx"] == 0].copy()
opens["a"] = opens["gap"].abs()
print("largest overnight gaps:")
print(opens.nlargest(8, "a")[["ts", "open", "gap"]].to_string())

largest moves inside a day:
                             ts   close     ret_1
91100 2025-04-07 10:10:00-04:00  187.71  0.039888
1721  2020-09-02 09:55:00-04:00  124.40 -0.033438
91102 2025-04-07 10:20:00-04:00  182.27 -0.031219
1876  2020-09-04 09:50:00-04:00  111.80 -0.030479
91318 2025-04-09 15:20:00-04:00  196.18  0.027754
1722  2020-09-02 10:00:00-04:00  127.85  0.027356
91293 2025-04-09 13:15:00-04:00  185.30  0.025858
9308  2021-01-25 11:10:00-05:00  137.06  0.024746
largest overnight gaps:
                              ts    open       gap
78216  2024-08-05 09:30:00-04:00  197.28 -0.098632
116682 2026-07-31 09:30:00-04:00  304.50 -0.088811
90936  2025-04-03 09:30:00-04:00  204.32 -0.084925
73338  2024-05-03 09:30:00-04:00  184.64  0.075225
89610  2025-03-11 09:30:00-04:00  222.28 -0.066398
91482  2025-04-14 09:30:00-04:00  210.15  0.064922
1950   2020-09-08 09:30:00-04:00  110.19 -0.063125
92964  2025-05-12 09:30:00-04:00  209.97  0.062341


## 9. Relative volume and trade count
IEX volume changes from year to year, so raw volume is not stable. Each bar is divided by the median of the same time-of-day over the previous 20 days (`shift(1)` keeps it past-only).

In [9]:
def past_median(s):
    return s.shift(1).rolling(20, min_periods=10).median()


for col in ["volume", "trade_count"]:
    ref = df.groupby("bar_idx")[col].transform(past_median)
    df["rel_" + col] = df[col] / ref

df[["ts", "rel_volume", "rel_trade_count"]].iloc[1000:1003]

,ts,rel_volume,rel_trade_count
1000,2020-08-19 14:50:00-04:00,1.424076,1.531250
1001,2020-08-19 14:55:00-04:00,1.456785,0.936170
1002,2020-08-19 15:00:00-04:00,0.424116,0.441176


## 10. Drop warm-up rows and save

In [10]:
before = len(df)
df = df.dropna(subset=["rel_volume", "rel_trade_count"]).reset_index(drop=True)
print("warm-up rows dropped:", before - len(df))

df["day"] = df["day"].astype(str)

df.to_parquet("aapl_5m_clean.parquet", index=False)
df.to_csv("aapl_5m_clean.csv", index=False)

print(df.shape)
print(df["timestamp"].min(), df["timestamp"].max())
print("rows with no 6-bar label (bars_left < 6):", (df["bars_left"] < 6).sum())
df.head(3)

warm-up rows dropped: 780
(120504, 19)
2020-08-17 13:30:00+00:00 2026-10-22 19:55:00+00:00
rows with no 6-bar label (bars_left < 6): 9300


,timestamp,open,high,low,close,volume,trade_count,vwap,ts,day,day_bars,is_half,bar_idx,bars_left,bar_end,gap,ret_1,rel_volume,rel_trade_count
0,2020-08-17 13:30:00+00:00,112.43,112.49,111.75,111.82,149688.0,437.0,112.169462,2020-08-17 09:30:00-04:00,2020-08-17,78,0,0,77,2020-08-17 13:35:00+00:00,0.009652,-0.005440,1.673614,2.009195
1,2020-08-17 13:35:00+00:00,111.83,112.09,111.76,111.86,109360.0,324.0,111.923282,2020-08-17 09:35:00-04:00,2020-08-17,78,0,1,76,2020-08-17 13:40:00+00:00,0.000000,0.000358,1.462149,2.339350
2,2020-08-17 13:40:00+00:00,111.84,111.94,111.67,111.82,72948.0,210.0,111.810012,2020-08-17 09:40:00-04:00,2020-08-17,78,0,2,75,2020-08-17 13:45:00+00:00,0.000000,-0.000358,2.467294,2.089552


## 11. Optional: cross-check against yfinance
yfinance gives about 60 days of 5-minute data. It uses full-market prices while the IEX feed uses one exchange, so a tiny difference (a few hundredths of a percent) is normal. A large difference is not.

In [11]:
import yfinance as yf

try:
    yf_df = yf.download("AAPL", period="60d", interval="5m",
                        auto_adjust=True, progress=False)
    yf_df.columns = [c[0] if isinstance(c, tuple) else c
                     for c in yf_df.columns]
    yf_df.index = yf_df.index.tz_convert("UTC")

    mine = df.set_index("timestamp")["close"]
    both = pd.concat([mine, yf_df["Close"]], axis=1, join="inner")
    both.columns = ["mine", "yf"]

    diff = (both["mine"] / both["yf"] - 1).abs()
    print("overlapping bars:", len(both))
    print("median diff:", diff.median())
    print("99th percentile diff:", diff.quantile(0.99))
    print("max diff:", diff.max())
except Exception as e:
    print("check skipped:", e)

ModuleNotFoundError: No module named 'yfinance'